# 03 — ETL GUB Barcelona

Construye `fact_incidentes_gub` a partir de los incidents gestionats por la Guàrdia Urbana de Barcelona (16 ficheros, 2010-2025), haciendo lookup de IDs contra las dimensiones del notebook 01.

**Fuente:** `data/raw/gub_barcelona/*.csv` (UTF-8, mensual, por barrio/distrito)

**Output:** `data/clean/fact_incidentes_gub.csv`

**Columnas objetivo:** `id, id_tiempo, id_territorio, id_tipo_incident, num_incidents`

**Reglas aplicadas:**
- P3 — Estandarizar nombres de columnas (2010-2015 espacios → 2016-2025 guiones bajos)
- P4 — `.str.strip()` en `Descripcio_Incident` (padded a 200 chars)
- P5 — Eliminar filas con `Codi_districte` null (realmente **710 filas** en 2024-2025: incidentes de emergencia tipo INCENDIS/EXPLOSIONS sin geolocalización + 1 fila basura).
- **2019 trae `Mes_any` como string** ('1','2'...) → se coerciona a int para el merge de tiempo
- **`Codi_Incident` es alfanumérico** (existe `'21M'`) → el lookup de tipo se hace como string
- `low_memory=False`, encoding UTF-8

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
GUB = RAW / 'gub_barcelona'

print('GUB path:', GUB)
print('Ficheros:', len(list(GUB.glob('*.csv'))))

---
## 1. Cargar y estandarizar los 16 ficheros (P3 + P4)

In [ ]:
RENAME_GUB = {
    'Codi Incident': 'Codi_Incident',
    'Descripció Incident': 'Descripcio_Incident',
    'Codi districte': 'Codi_districte',
    'Nom districte': 'Nom_districte',
    'Codi barri': 'Codi_barri',
    'Nom barri': 'Nom_barri',
    'NK Any': 'NK_Any',
    'Mes de any': 'Mes_any',
    'Nom mes': 'Nom_mes',
    "Número d'incidents GUB": 'Numero_incidents_GUB'
}

dfs = []
for f in sorted(GUB.glob('*.csv')):
    df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df = df.rename(columns=RENAME_GUB)
    df['Descripcio_Incident'] = df['Descripcio_Incident'].str.strip()  # P4
    dfs.append(df)
    print(f'{f.name}: {df.shape[0]} filas, {df.shape[1]} cols')

gub = pd.concat(dfs, ignore_index=True)
print(f'\nTotal concatenado: {gub.shape}')
print('Columnas:', gub.columns.tolist())

In [ ]:
print('Nulls por columna:')
print(gub.isnull().sum())
print('\nRango años (NK_Any):', sorted(gub['NK_Any'].unique()))
print('Descripcions únicas tras strip:', gub['Descripcio_Incident'].nunique())

---
## 2. Eliminar filas sin distrito (P5)

In [ ]:
filas_antes = len(gub)
# Inspeccionar lo que se va a eliminar
sin_districte = gub[gub['Codi_districte'].isnull()]
print(f'Filas con Codi_districte null: {len(sin_districte)}')
if len(sin_districte) > 0:
    print('Años afectados:', sorted(sin_districte['NK_Any'].unique()))
    print('Descripcions:', sin_districte['Descripcio_Incident'].unique())

gub = gub.dropna(subset=['Codi_districte']).copy()
print(f'\nFilas eliminadas: {filas_antes - len(gub)}')
print(f'Shape tras limpiar: {gub.shape}')
print('Nulls restantes en Codi_barri:', gub['Codi_barri'].isnull().sum())

In [ ]:
# Normalizar códigos de barrio/distrito a int (vienen como float por los nulls eliminados)
gub['Codi_districte'] = gub['Codi_districte'].astype(int)
gub['Codi_barri'] = gub['Codi_barri'].astype(int)

# Normalizar año/mes a int — OJO: 2019 trae Mes_any como STRING ('1','2'...),
# lo que rompe el merge con dim_tiempo (mes int). Se coerciona toda la columna.
gub['NK_Any'] = gub['NK_Any'].astype(int)
gub['Mes_any'] = pd.to_numeric(gub['Mes_any'], errors='coerce').astype('Int64')
n_mes_nulos = gub['Mes_any'].isnull().sum()
print('Meses no numéricos tras coerción:', n_mes_nulos)
assert n_mes_nulos == 0, 'Hay valores de Mes_any que no son numéricos — revisar'
gub['Mes_any'] = gub['Mes_any'].astype(int)

print('Códigos y fechas normalizados a int')
print('Distritos únicos:', sorted(gub['Codi_districte'].unique()))
print('Meses únicos:', sorted(gub['Mes_any'].unique()))

---
## 3. Cargar dimensiones

In [ ]:
dim_tiempo = pd.read_csv(CLEAN / 'dim_tiempo.csv', encoding='utf-8')
dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
dim_tipo_delito = pd.read_csv(CLEAN / 'dim_tipo_delito.csv', encoding='utf-8')

print('dim_tiempo:', dim_tiempo.shape)
print('dim_territorio:', dim_territorio.shape)
print('dim_tipo_delito:', dim_tipo_delito.shape)

---
## 4. Lookup id_tiempo (por NK_Any + Mes_any)

In [ ]:
gub = gub.merge(
    dim_tiempo[['id_tiempo', 'anyo', 'mes']],
    left_on=['NK_Any', 'Mes_any'],
    right_on=['anyo', 'mes'],
    how='left'
)
n_sin_tiempo = gub['id_tiempo'].isnull().sum()
print(f'Filas sin id_tiempo: {n_sin_tiempo}')
if n_sin_tiempo > 0:
    print(gub[gub['id_tiempo'].isnull()][['NK_Any', 'Mes_any']].drop_duplicates())

---
## 5. Lookup id_territorio (por cod_barri + cod_districte)

Se filtra `dim_territorio` a `fuente == 'gub'`. Los códigos de barrio en la dimensión vienen como float (por los nulls de otras fuentes) → se normalizan a int para el match.

In [ ]:
terr_gub = dim_territorio[dim_territorio['fuente'] == 'gub'][['id_territorio', 'cod_barri', 'cod_districte']].copy()
terr_gub['cod_barri'] = terr_gub['cod_barri'].astype(int)
terr_gub['cod_districte'] = terr_gub['cod_districte'].astype(int)
print('Territorios GUB en dimensión:', len(terr_gub))
print('Combinaciones barri/districte únicas en datos:', gub[['Codi_barri', 'Codi_districte']].drop_duplicates().shape[0])

gub = gub.merge(
    terr_gub,
    left_on=['Codi_barri', 'Codi_districte'],
    right_on=['cod_barri', 'cod_districte'],
    how='left'
)
n_sin_terr = gub['id_territorio'].isnull().sum()
print(f'\nFilas sin id_territorio: {n_sin_terr}')
if n_sin_terr > 0:
    print(gub[gub['id_territorio'].isnull()][['Codi_barri', 'Nom_barri', 'Codi_districte']].drop_duplicates())

---
## 6. Lookup id_tipo_incident (por Codi_Incident + Descripcio_Incident)

Se filtra `dim_tipo_delito` a `fuente == 'gub'` y se hace match sobre `(codigo, descripcio)`.

In [ ]:
# OJO: Codi_Incident es alfanumérico (existe el valor '21M'), NO se puede castear a int.
# Se hace el match como string en ambos lados.
tipo_gub = dim_tipo_delito[dim_tipo_delito['fuente'] == 'gub'][['id_tipo_delito', 'codigo', 'descripcio']].copy()
tipo_gub['codigo'] = tipo_gub['codigo'].astype(str)
print('Tipos GUB en dimensión:', len(tipo_gub))

gub['Codi_Incident'] = gub['Codi_Incident'].astype(str)

gub = gub.merge(
    tipo_gub,
    left_on=['Codi_Incident', 'Descripcio_Incident'],
    right_on=['codigo', 'descripcio'],
    how='left'
)
gub = gub.rename(columns={'id_tipo_delito': 'id_tipo_incident'})
n_sin_tipo = gub['id_tipo_incident'].isnull().sum()
print(f'\nFilas sin id_tipo_incident: {n_sin_tipo}')
if n_sin_tipo > 0:
    print(gub[gub['id_tipo_incident'].isnull()][['Codi_Incident', 'Descripcio_Incident']].drop_duplicates())

---
## 7. Construir tabla final

In [ ]:
fact = gub.rename(columns={'Numero_incidents_GUB': 'num_incidents'})[
    ['id_tiempo', 'id_territorio', 'id_tipo_incident', 'num_incidents']
].copy()

# Verificar que no hay lookups fallidos antes de convertir a int
assert fact[['id_tiempo', 'id_territorio', 'id_tipo_incident']].isnull().sum().sum() == 0, 'Hay lookups fallidos — revisar arriba'
for c in ['id_tiempo', 'id_territorio', 'id_tipo_incident']:
    fact[c] = fact[c].astype(int)

# num_incidents viene como float (por el null de la fila basura ya eliminada) → a int
assert fact['num_incidents'].isnull().sum() == 0, 'Hay num_incidents nulos'
fact['num_incidents'] = fact['num_incidents'].astype(int)

# PK incremental
fact.insert(0, 'id', range(1, len(fact) + 1))

print('Shape final:', fact.shape)
print('\nTipos de datos:')
print(fact.dtypes)
fact.head()

In [ ]:
print('Validaciones:')
print('  Nulls totales:', fact.isnull().sum().sum())
print('  Filas duplicadas (sin id):', fact.drop(columns='id').duplicated().sum())
print('  Suma num_incidents:', fact['num_incidents'].sum())
print('  Rango num_incidents:', fact['num_incidents'].min(), '-', fact['num_incidents'].max())
print('  id_tiempo en rango [1,192]:', fact['id_tiempo'].between(1,192).all())

---
## 8. Guardar

In [ ]:
ruta_salida = CLEAN / 'fact_incidentes_gub.csv'
fact.to_csv(ruta_salida, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta_salida.name}: {len(fact)} filas')
print('\nListo para continuar con 04_etl_ministerio_ine.ipynb')